# Task 2 - NLP Sentiment Analysis Pipeline
**Datheon Machine Learning & Data Science Internship**

Pipeline: Load -> Preprocess (HTML, stopwords, lemmatization) -> TF-IDF (max_features=5000) -> MultinomialNB + LogisticRegression -> Metrics -> predict_sentiment()

## 1. Imports & Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import re, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix)
for c in ["stopwords","wordnet","punkt","punkt_tab","omw-1.4"]: nltk.download(c, quiet=True)
BG,AX="#0f0f1a","#1a1a2e"; C1,C2,C3,TX,GR="#00d4ff","#a855f7","#22c55e","#e2e8f0","#2d2d4e"
plt.rcParams.update({"figure.facecolor":BG,"axes.facecolor":AX,"axes.edgecolor":GR,
    "axes.labelcolor":TX,"axes.titlecolor":TX,"xtick.color":TX,"ytick.color":TX,
    "text.color":TX,"grid.color":GR,"grid.linewidth":0.5,"axes.grid":True,
    "legend.facecolor":AX,"legend.edgecolor":GR,"savefig.facecolor":BG,"savefig.dpi":180})
OUT = Path(".")
print("Setup complete.")

## 2. Load Dataset

In [ ]:
df_raw = pd.read_csv("sentiment_reviews.csv")
print("Shape:", df_raw.shape)
display(df_raw)
print(df_raw["sentiment"].value_counts())

## 3. Text Preprocessing

In [ ]:
STOP = set(stopwords.words("english")); LEM = WordNetLemmatizer()
def preprocess(text):
    text = re.sub(r"<[^>]+>"," ",text)
    text = text.lower()
    text = re.sub(r"http\S+"," ",text)
    text = re.sub(r"[^a-z\s]"," ",text)
    toks = word_tokenize(text)
    return " ".join([LEM.lemmatize(t) for t in toks if t not in STOP and len(t)>2])
print("preprocess defined.")

## 4. Dataset Augmentation (10 to 100 samples)

In [ ]:
EXTRA = [
    ("This is hands down the best purchase I have ever made!","positive"),
    ("Incredible quality, fast shipping, would buy again.","positive"),
    ("My whole family loves it. Five stars without hesitation.","positive"),
    ("The build quality is superb and it looks gorgeous.","positive"),
    ("Works exactly as advertised. Very satisfied customer!","positive"),
    ("Delivery was prompt and the packaging was excellent.","positive"),
    ("Outstanding performance for the price, truly impressive.","positive"),
    ("I am completely satisfied. Would strongly recommend to anyone.","positive"),
    ("Top-notch product. Customer support was also very helpful.","positive"),
    ("Amazing! This exceeded every single one of my expectations.","positive"),
    ("Perfect gift idea. Recipient was absolutely thrilled.","positive"),
    ("Love the design and how lightweight it is. Great product!","positive"),
    ("Setup was easy and it works flawlessly right out of the box.","positive"),
    ("Premium feel without the premium price tag. Highly recommend!","positive"),
    ("Exactly what I needed. Fast delivery and great value.","positive"),
    ("Brilliant product, very intuitive to use.","positive"),
    ("I am impressed by the durability. Survived a drop test!","positive"),
    ("Wonderful experience from purchase to delivery.","positive"),
    ("This product changed my daily routine for the better.","positive"),
    ("Absolutely love it. Already recommended to three friends.","positive"),
    ("Great value for money. Very pleased with the purchase.","positive"),
    ("The quality is unmatched at this price point.","positive"),
    ("Quick shipping and the item arrived in perfect condition.","positive"),
    ("Fantastic product! Would not hesitate to buy again.","positive"),
    ("This is exactly what I was looking for.","positive"),
    ("Excellent craftsmanship. Attention to detail is remarkable.","positive"),
    ("So happy with this purchase. Makes life so much easier.","positive"),
    ("Best seller for a reason! Lives up to the hype.","positive"),
    ("I was skeptical at first but it really is that good.","positive"),
    ("Solid product, great instructions, fantastic result.","positive"),
    ("100 percent satisfied. This is now my go-to product.","positive"),
    ("One of the best online purchases I have made in years.","positive"),
    ("The colors are vibrant and the material feels very premium.","positive"),
    ("Incredible customer service when I had a question.","positive"),
    ("Exactly as described in the listing. Very happy customer.","positive"),
    ("This product is a gem. Highly recommend without reservation.","positive"),
    ("Could not be happier. Great product at a fair price.","positive"),
    ("My expectations were high and they were still exceeded.","positive"),
    ("Well built, stylish, and very functional. Love it!","positive"),
    ("Delivery was faster than expected and product is perfect.","positive"),
    ("This is a game changer. I use it every single day.","positive"),
    ("Pleasantly surprised by the quality. Very happy!","positive"),
    ("Easy to use and works great. Exactly what I needed.","positive"),
    ("This is simply perfect. Bought two already!","positive"),
    ("Superb product, smooth experience, great value!","positive"),
    ("Completely useless. Broke after one hour of use.","negative"),
    ("Worst purchase I have made. Total waste of money.","negative"),
    ("Item arrived broken and customer support was no help.","negative"),
    ("Nothing like the photos. Very misleading description.","negative"),
    ("Poor build quality. Fell apart within a week.","negative"),
    ("I am deeply disappointed. This does not work as advertised.","negative"),
    ("Returned immediately. Product was defective on arrival.","negative"),
    ("Terrible quality. The material feels extremely cheap.","negative"),
    ("Do not buy this. It stopped functioning after two uses.","negative"),
    ("Huge disappointment. Expected so much more for this price.","negative"),
    ("The seller ignored all my complaints. Awful experience.","negative"),
    ("Packaging was damaged and item inside was scratched.","negative"),
    ("Looks nothing like the pictures. Extremely misleading.","negative"),
    ("This product is a scam. Avoid at all costs.","negative"),
    ("Did not last even a week. Very poor durability.","negative"),
    ("Completely useless for the intended purpose.","negative"),
    ("I am furious. This broke on the very first day.","negative"),
    ("Customer service was rude and unhelpful. Never buying again.","negative"),
    ("One star is too generous for this terrible product.","negative"),
    ("Worst quality I have ever seen. Absolute garbage.","negative"),
    ("Shipping took three weeks and the item came crushed.","negative"),
    ("Nothing works properly. Extremely frustrated with this.","negative"),
    ("Returned it after one day. It simply does not function.","negative"),
    ("I feel cheated. The product is nowhere near what was shown.","negative"),
    ("Horribly made. Would not recommend to anyone.","negative"),
    ("Stopped working after the first week. Very unreliable.","negative"),
    ("Defective out of the box and returns process is a nightmare.","negative"),
    ("Poor instructions and even poorer build. Avoid!","negative"),
    ("Total letdown. The description was completely inaccurate.","negative"),
    ("Broke during normal use within days. Very disappointing.","negative"),
    ("I expected better given the price. Very unsatisfied.","negative"),
    ("Flimsy construction, looks nothing like the ad photos.","negative"),
    ("This was a complete waste of my hard-earned money.","negative"),
    ("Product failed immediately. Would not recommend.","negative"),
    ("Extremely disappointed. This product is a money grab.","negative"),
    ("Terrible experience from order to delivery. Never again.","negative"),
    ("Item was missing parts and customer service did nothing.","negative"),
    ("Honestly disgusted with this purchase. Total failure.","negative"),
    ("Product looked good in pics but reality is very different.","negative"),
    ("Stopped working permanently after minimal use. Junk.","negative"),
    ("It is okay. Does the job but nothing special.","neutral"),
    ("Decent product for the price. Not bad but not great either.","neutral"),
    ("Average quality. Met my basic expectations nothing more.","neutral"),
    ("It works as described. Fairly standard product overall.","neutral"),
    ("Nothing remarkable about this but it does what it says.","neutral"),
]
df_e = pd.DataFrame(EXTRA, columns=["review_text","sentiment"])
df_e["review_id"] = range(df_raw["review_id"].max()+1, df_raw["review_id"].max()+1+len(df_e))
df = pd.concat([df_raw[["review_id","review_text","sentiment"]], df_e], ignore_index=True)
df["clean"] = df["review_text"].apply(preprocess)
print("Total samples:", len(df))
print(df["sentiment"].value_counts())

## 5. EDA

In [ ]:
df["tokens"] = df["clean"].apply(lambda x: len(x.split()))
CLR={"positive":C3,"negative":"#ef4444","neutral":C1}; ORD=["positive","negative","neutral"]
fig,axes=plt.subplots(1,2,figsize=(14,5))
fig.suptitle("EDA - Sentiment Dataset",fontsize=15,fontweight="bold")
ax=axes[0]; cnt=df["sentiment"].value_counts().reindex(ORD)
bars=ax.bar(ORD,cnt.values,color=[CLR[l] for l in ORD],width=0.5,edgecolor=BG)
[ax.text(b.get_x()+b.get_width()/2,b.get_height()+.5,str(v),ha="center",va="bottom",fontsize=12,fontweight="bold") for b,v in zip(bars,cnt.values)]
ax.set_title("Class Distribution"); ax.set_ylabel("Count")
ax=axes[1]
[ax.hist(df[df["sentiment"]==l]["tokens"],bins=12,alpha=0.7,color=CLR[l],label=l,edgecolor=BG) for l in ORD]
ax.set_title("Token Count per Review"); ax.set_xlabel("Tokens"); ax.legend()
plt.tight_layout(); plt.savefig(OUT/"eda_distribution.png",bbox_inches="tight"); plt.show()
print("Saved eda_distribution.png")

## 6. TF-IDF + Train/Test Split

In [ ]:
LM={"positive":0,"negative":1,"neutral":2}; LI={v:k for k,v in LM.items()}; TN=["positive","negative","neutral"]
X=df["clean"].values; y=df["sentiment"].map(LM).values
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
vec=TfidfVectorizer(max_features=5000,ngram_range=(1,2),min_df=1)
Xtr_v=vec.fit_transform(Xtr); Xte_v=vec.transform(Xte)
print("Train:", Xtr_v.shape, "  Test:", Xte_v.shape, "  Vocab:", len(vec.vocabulary_))

## 7. Train Models

In [ ]:
nb=MultinomialNB(alpha=0.5); nb.fit(Xtr_v,ytr)
lr=LogisticRegression(max_iter=1000,C=1.0,solver="lbfgs",random_state=42); lr.fit(Xtr_v,ytr)
print("Both models trained.")

## 8. Evaluation - Classification Reports

In [ ]:
def evaluate(model,Xv,yt,name):
    p=model.predict(Xv)
    r=dict(model=name,
           accuracy=accuracy_score(yt,p),
           precision=precision_score(yt,p,average="weighted",zero_division=0),
           recall=recall_score(yt,p,average="weighted",zero_division=0),
           f1=f1_score(yt,p,average="weighted",zero_division=0),preds=p)
    print("\n--", name)
    for k in ["accuracy","precision","recall","f1"]: print("  %s: %.4f" % (k,r[k]))
    print(classification_report(yt,p,target_names=TN,zero_division=0))
    return r
nb_m=evaluate(nb,Xte_v,yte,"Multinomial Naive Bayes")
lr_m=evaluate(lr,Xte_v,yte,"Logistic Regression")

## 9. Cross-Validation

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42); Xa=vec.transform(X)
nb_cv=cross_val_score(nb,Xa,y,cv=cv,scoring="f1_weighted")
lr_cv=cross_val_score(lr,Xa,y,cv=cv,scoring="f1_weighted")
print("NB  CV F1: %.4f +/- %.4f  folds:" % (nb_cv.mean(),nb_cv.std()), np.round(nb_cv,4))
print("LR  CV F1: %.4f +/- %.4f  folds:" % (lr_cv.mean(),lr_cv.std()), np.round(lr_cv,4))

## 10. Confusion Matrices

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
fig.suptitle("Confusion Matrices - Sentiment Classification",fontsize=14,fontweight="bold")
for ax,(name,preds,col) in zip(axes,[("Multinomial Naive Bayes",nb_m["preds"],C1),("Logistic Regression",lr_m["preds"],C2)]):
    cm=confusion_matrix(yte,preds,normalize="true")
    sns.heatmap(cm,annot=True,fmt=".2f",ax=ax,cmap="Blues",linewidths=0.5,
                xticklabels=TN,yticklabels=TN,cbar_kws={"shrink":0.8})
    ax.set_title(name,fontsize=12,color=col); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.savefig(OUT/"confusion_matrices.png",bbox_inches="tight"); plt.show()
print("Saved confusion_matrices.png")

## 11. Benchmark Bar Chart

In [ ]:
mc=["accuracy","precision","recall","f1"]
mdf=pd.DataFrame([{k:v for k,v in nb_m.items() if k!="preds"},{k:v for k,v in lr_m.items() if k!="preds"}]).set_index("model")
x=np.arange(len(mc)); w=0.35
fig,ax=plt.subplots(figsize=(12,6))
fig.suptitle("Model Benchmark - Sentiment Classification",fontsize=14,fontweight="bold")
b1=ax.bar(x-w/2,mdf.loc["Multinomial Naive Bayes",mc],w,label="Naive Bayes",color=C1,edgecolor=BG)
b2=ax.bar(x+w/2,mdf.loc["Logistic Regression",mc],w,label="Logistic Regression",color=C2,edgecolor=BG)
for bars in [b1,b2]:
    for b in bars: h=b.get_height(); ax.text(b.get_x()+b.get_width()/2,h+.005,"%.3f"%h,ha="center",va="bottom",fontsize=9,fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(["Accuracy","Precision","Recall","F1-Score"],fontsize=11)
ax.set_ylim(0,1.15); ax.set_ylabel("Score"); ax.legend(fontsize=11)
plt.tight_layout(); plt.savefig(OUT/"metrics_comparison.png",bbox_inches="tight"); plt.show()
print("Saved metrics_comparison.png"); display(mdf[mc].round(4))

## 12. Top TF-IDF Features per Sentiment Class

In [ ]:
fn=vec.get_feature_names_out(); N=15
fig,axes=plt.subplots(1,3,figsize=(18,6))
fig.suptitle("Top TF-IDF Features per Sentiment Class",fontsize=13,fontweight="bold")
for ci,(ax,lbl,col) in enumerate(zip(axes,TN,[C3,"#ef4444",C1])):
    lp=nb.feature_log_prob_[ci]; ti=np.argsort(lp)[-N:][::-1]
    ax.barh(range(N),lp[ti][::-1],color=col,edgecolor=BG,height=0.7)
    ax.set_yticks(range(N)); ax.set_yticklabels([fn[i] for i in ti][::-1],fontsize=9)
    ax.set_title(lbl.capitalize(),fontsize=12,color=col); ax.set_xlabel("Log Prob"); ax.invert_yaxis()
plt.tight_layout(); plt.savefig(OUT/"top_features.png",bbox_inches="tight"); plt.show()
print("Saved top_features.png")

## 13. predict_sentiment() - Live Prediction Helper

In [ ]:
def predict_sentiment(text, model=None, verbose=True):
    """Predict sentiment for any raw review text."""
    if model is None: model = lr
    cleaned = preprocess(text)
    v = vec.transform([cleaned])
    pid = model.predict(v)[0]; lbl = LI[pid]
    probs = model.predict_proba(v)[0] if hasattr(model,"predict_proba") else [0.0]*3
    conf = probs[pid]
    pd_ = {TN[i]:round(float(p),4) for i,p in enumerate(probs)}
    if verbose:
        tag={"positive":"POSITIVE [+]","negative":"NEGATIVE [-]","neutral":"NEUTRAL [~]"}
        print("="*58)
        print("  Input  :", repr(text[:60]+"..." if len(text)>60 else text))
        print("  Cleaned:", repr(cleaned[:60]+"..." if len(cleaned)>60 else cleaned))
        print("-"*58)
        print("  Result :", tag[lbl], "  (%.1f%% confident)" % (conf*100))
        print("-"*58)
        for l,p in sorted(pd_.items(),key=lambda x:-x[1]):
            b = "#"*int(p*25)+"."*(25-int(p*25))
            print("  %-10s [%s] %.2f%%" % (l, b, p*100))
        print("="*58)
    return dict(label=lbl,label_id=int(pid),confidence=float(conf),probabilities=pd_)
print("predict_sentiment() ready!")

## 14. Live Demo Predictions

In [ ]:
demos = [
    "This product is absolutely amazing! I love everything about it.",
    "Terrible experience. The item arrived broken and support was useless.",
    "It is okay I guess. Does the job but nothing special.",
    "Best purchase of the year. Highly recommend!",
    "Complete waste of money. Would never buy again.",
    "Average product. Not impressed but not disappointed either.",
]
for d in demos:
    predict_sentiment(d); print()

### Try Your Own Input

In [ ]:
custom = "The packaging was nice but the product itself was really underwhelming."
result = predict_sentiment(custom)
print("Result:", result)